# Bahasa Malaysia spellchecker — GECToR fine-tune of LFM2.5-Encoder-350M

Drives the `lfm_my` package end-to-end (data → train → probe → export) on a free Colab T4.
Checkpoints and data persist on Google Drive, so a disconnected runtime resumes where it left off.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.isdir('/content/lfm-my'):
    !git clone https://github.com/GITHUB_USER/lfm-my.git /content/lfm-my
else:
    !(cd /content/lfm-my && git pull)
!pip install -q -e /content/lfm-my

## Config — every knob in one place

In [ ]:
import json, math, random, time
from pathlib import Path
import torch

ENCODER = "LiquidAI/LFM2.5-Encoder-350M"
DATA_DIR = Path("/content/drive/MyDrive/lfm-my/data")     # persists across sessions
CKPT_DIR = Path("/content/drive/MyDrive/lfm-my/ckpts")
EXPORT_DIR = Path("/content/drive/MyDrive/lfm-my/export")
MAX_LEN, BATCH, ACCUM = 128, 8, 4
LR, WARMUP, WD = 2e-5, 0.1, 0.1
EPOCHS, PATIENCE = 7, 2
CKPT_EVERY = 500                                            # optimizer steps
WIKI_LINES = 60_000                                         # cap the stream; see data cell
SEED = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
# The free Colab GPU is a T4 (Turing, sm_75), which has NO bfloat16 -- that needs Ampere or
# newer. Pick the dtype from the hardware, and keep GradScaler for the fp16 path (bf16 does
# not need loss scaling).
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
print(f"device={DEVICE} amp={USE_AMP} dtype={AMP_DTYPE}")
torch.manual_seed(SEED); random.seed(SEED)

## Data

Generate ~15–20K train + ~1K val pairs from Malay Wikipedia (first run only; cached on Drive).

`load_wiki_sentences` streams, so it is capped with `WIKI_LINES`: materialising all of Malay
Wikipedia into a list would exhaust the Colab instance's RAM long before training starts.

In [ ]:
from lfm_my.data_build import build_from_lines, load_wiki_sentences
from lfm_my.errors import InjectorConfig

if not (DATA_DIR / "ms-train.jsonl").exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    lines = list(load_wiki_sentences("ms", limit=WIKI_LINES))
    print(f"wiki lines: {len(lines)}")
    stats = build_from_lines(lines, DATA_DIR / "ms",
                             InjectorConfig(seed=SEED), pairs_per_sentence=1,
                             val_size=1000, seed=SEED)
    print(stats)
else:
    print("data already on Drive, skipping build")

In [ ]:
import itertools
for line in itertools.islice(open(DATA_DIR / "ms-train.jsonl"), 5):
    p = json.loads(line)
    print("NOISY   :", p["noisy"])
    print("CORRECT :", p["correct"], "\n")

## Model

In [ ]:
from transformers import AutoTokenizer
from lfm_my.model import build_tagger

tok = AutoTokenizer.from_pretrained(ENCODER, trust_remote_code=True)
V = len(tok.get_vocab())
print("vocab:", V)

model = build_tagger(ENCODER).to(DEVICE)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

### Tokenization consistency check

Training aligns edits at the **word** level, so `convert_pair` encodes one word at a time.
Inference (`model.correct()`) encodes the **whole sentence** in one call. For tokenizers that
mark word boundaries by a leading space (GPT-2 style), those two disagree and the model would
be served inputs shaped differently from the ones it trained on. This cell reports which case
this tokenizer is in — read the warning before spending a training run.

In [ ]:
from lfm_my.convert import pieces_of

enc_word = lambda w: tok.encode(w, add_special_tokens=False)
probe_text = "dia pergi ke sekolah setiap hari"
per_word = pieces_of(probe_text.split(), enc_word)
whole = tok.encode(probe_text, add_special_tokens=False)
TOKENIZATION_CONSISTENT = per_word == whole

print("per-word == whole-sentence:", TOKENIZATION_CONSISTENT)
if not TOKENIZATION_CONSISTENT:
    print()
    print("  WARNING: this tokenizer is context-sensitive.")
    print(f"    per-word : {per_word}")
    print(f"    whole    : {whole}")
    print("  Training inputs (per-word) will not match what model.correct() feeds the model at")
    print("  inference. Fix before training: either encode words with a leading space in")
    print("  GecDataset/convert_pair, or align convert_pair on whole-sentence pieces.")

## Datasets and loaders

In [ ]:
from functools import partial
from torch.utils.data import DataLoader
from lfm_my.dataset import GecDataset, collate_fn

def load_pairs(name):
    return [json.loads(l) for l in open(DATA_DIR / f"ms-{name}.jsonl")]

train_ds = GecDataset(load_pairs("train"), tok, V, max_len=MAX_LEN)
val_ds = GecDataset(load_pairs("val"), tok, V, max_len=MAX_LEN)
print(f"train samples: {len(train_ds)} (skipped {train_ds.skipped}), val: {len(val_ds)}")

pad = tok.pad_token_id if tok.pad_token_id is not None else 0
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      collate_fn=partial(collate_fn, pad_id=pad))
val_dl = DataLoader(val_ds, batch_size=BATCH, collate_fn=partial(collate_fn, pad_id=pad))

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from lfm_my.train import save_checkpoint, load_checkpoint

opt = AdamW(model.parameters(), lr=LR, weight_decay=WD)
total_steps = max(1, (len(train_dl) // ACCUM) * EPOCHS)
warm = int(total_steps * WARMUP)

def lr_at(step):
    if step < warm:
        return max(1e-2, step / max(1, warm))
    prog = (step - warm) / max(1, total_steps - warm)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

sched = LambdaLR(opt, lr_at)
scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP and AMP_DTYPE == torch.float16)
print(f"total_steps={total_steps} warmup={warm}")

## Train

In [ ]:
from lfm_my.train import aggregate_metrics, compute_loss, metrics_for

STATE = {"step": 0, "best_val": float("inf")}

def run_epoch(loader, train: bool):
    model.train(train)
    loss_sum, n_seqs, mets = 0.0, 0, []
    opt.zero_grad()
    for i, batch in enumerate(loader):
        b = {k: v.to(DEVICE) for k, v in batch.items()}
        # no_grad on the eval pass: building the graph for validation wastes T4 memory
        with torch.set_grad_enabled(train):
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                out = model(b["input_ids"], b["attention_mask"])
                # NB: name the detect loss `det_loss`, not `dl` -- `dl` would shadow the loader
                loss, lab_loss, det_loss = compute_loss(
                    out, b["label_targets"], b["detect_targets"])
        if train:
            scaler.scale(loss / ACCUM).backward()
            if (i + 1) % ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
                STATE["step"] += 1
                if STATE["step"] % CKPT_EVERY == 0:        # Colab-timeout safety
                    save_checkpoint(CKPT_DIR / "ckpt.pt", model, opt,
                                    STATE["step"], STATE["best_val"])
                    print(f"  [ckpt] step {STATE['step']} saved", flush=True)
        bs = b["input_ids"].size(0)
        loss_sum += float(loss.detach()) * bs
        n_seqs += bs
        # Pool with aggregate_metrics, never a plain mean: an all-$KEEP batch has no non-KEEP
        # target to score and reports NaN, which a naive average would either poison or inflate.
        mets.append(metrics_for(out, b["label_targets"], b["detect_targets"]))
        if train and (i + 1) % 50 == 0:
            agg = aggregate_metrics(mets)
            print(f"  step {i+1}/{len(loader)} loss={loss_sum/max(1, n_seqs):.4f} "
                  f"lab_acc={agg['label_acc_nokeep']:.3f}", flush=True)
    return {"loss": loss_sum / max(1, n_seqs), **aggregate_metrics(mets)}

In [ ]:
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if (CKPT_DIR / "ckpt.pt").exists():
    meta = load_checkpoint(CKPT_DIR / "ckpt.pt", model, opt)
    STATE.update(step=meta["step"], best_val=meta["best_val"])
    print(f"resumed from step {STATE['step']}, best_val={STATE['best_val']:.4f}")

bad = 0
for epoch in range(EPOCHS):
    t0 = time.time()
    tr = run_epoch(train_dl, train=True)
    va = run_epoch(val_dl, train=False)
    print(f"epoch {epoch+1}: train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} "
          f"val_lab_acc={va['label_acc_nokeep']:.3f} val_det_acc={va['detect_acc']:.3f} "
          f"({time.time()-t0:.0f}s)", flush=True)
    if va["loss"] < STATE["best_val"] - 1e-4:
        STATE["best_val"], bad = va["loss"], 0
        save_checkpoint(CKPT_DIR / "best.pt", model, opt, STATE["step"], STATE["best_val"])
        print("  new best — saved best.pt")
    else:
        bad += 1
        if bad >= PATIENCE:
            print("early stopping"); break

## Probe — spot-check the tagger on known Malay errors

In [ ]:
from lfm_my.text import tokenize

load_checkpoint(CKPT_DIR / "best.pt", model)
model.eval()
PROBES = [
    "saya sudah makan nasik di kedai itu semalam",
    "dia tidak arah pinjam buku",
    "buku itu di baca oleh ali",
    "dia pergi ke sekolah setiap hari",
]
bos = tok.bos_token_id if tok.bos_token_id is not None else 1
with torch.no_grad():
    for p in PROBES:
        src = tokenize(p)
        # encode per WORD, matching how GecDataset built the training inputs
        pieces = pieces_of(src.split(), enc_word)[:MAX_LEN - 1]
        ids = torch.tensor([[bos] + pieces]).to(DEVICE)
        out = model(ids, torch.ones_like(ids))
        preds = out["label_logits"].argmax(-1)[0]
        err = out["detect_logits"].softmax(-1)[0][:, 1]
        edits = [int(x) for x in (preds != 0).nonzero().flatten()]
        print("IN   :", p)
        print("tags :", [(int(i), int(preds[i]), round(float(err[i]), 3)) for i in edits] or "KEEP all")
        print()

## Export

In [ ]:
from lfm_my.export import export_model_dir
export_model_dir(model, tok, EXPORT_DIR, encoder_name=ENCODER)
print(sorted(p.name for p in EXPORT_DIR.iterdir()))

## Next steps

Download `EXPORT_DIR`, or push it straight to the Hub:

```python
from lfm_my.export import push_model_repo
push_model_repo(EXPORT_DIR, "USER/lfm-malay-spellchecker")
```

Then verify the app locally (Task 12) and publish the Space (Task 13).